In [1]:
import pandas as pd

In [116]:
import folium

# Create a map centered at the specified coordinates
map_center = [-7.75, 108.00]
map_ = folium.Map(location=map_center, zoom_start=6)

# Add a marker for the location
folium.Marker(
    location=map_center,
    popup="2017 Java Earthquake\nLat: -7.75, Long: 108.00",
    icon=folium.Icon(color="red", icon="info-sign")
).add_to(map_)

# Display the map
map_


In [102]:
df1=pd.read_csv('indo_ips_lat_lon.csv')

In [111]:
df2=pd.read_csv('country.csv')

In [112]:
df2.head()

,start_ip,end_ip,country,country_name,continent,continent_name
0,1.0.0.0,1.0.0.255,AU,Australia,OC,Oceania
1,1.0.1.0,1.0.3.255,CN,China,AS,Asia
2,1.0.4.0,1.0.7.255,AU,Australia,OC,Oceania
3,1.0.8.0,1.0.15.255,CN,China,AS,Asia
4,1.0.16.0,1.0.31.255,JP,Japan,AS,Asia


In [115]:
df2[df2['country']=='ID'].shape

(15957, 6)

In [110]:
len(df1.IP.unique())

898

In [ ]:
df.longitude

In [95]:
len(df1.NS_ID.unique())

38810

In [72]:
df=pd.read_csv('nss_loc.csv')

In [73]:
df.shape

(23467, 18)

In [66]:
df.columns

Index(['NS_ID', '_c0', 'IP', 'latitude', 'longitude', 'ID', 'SEEN',
       'AVG_RTT_MILLIS', 'IS_ONLINE', 'IS_AUTH', 'SERIAL', 'VERSION', 'EDNS0',
       'CNAME', 'BOGUS', 'VALID_SOA', 'KEYS_VERIF', 'FOUND_WHERE'],
      dtype='object')

In [ ]:
df=df.drop(columns='_c0')

In [ ]:
df=df[['NS_ID', 'Ve', 'IP','latitude','longitude','SEEN','IS_ONLINE','IS_AUTH']]

In [71]:
df.head()

,NS_ID,CNAME,IP,latitude,longitude,SEEN,IS_ONLINE,IS_AUTH
0,32200887,0,119.235.255.215,-6.2146,106.8451,1498103754,1,1
1,32200892,0,119.235.255.215,-6.2146,106.8451,1498103754,1,1
2,32200905,0,119.235.255.215,-6.2146,106.8451,1498103756,1,1
3,32200912,0,119.235.255.215,-6.2146,106.8451,1498103756,1,1
4,32200917,0,119.235.255.215,-6.2146,106.8451,1498103756,1,1


In [7]:
df['LAST_SEEN'] = pd.to_datetime(df['LAST_SEEN'], unit='s')

In [9]:
len(df.NAME.unique())

166

In [10]:
df.groupby('NS_ID')

In [10]:
df['IS_ONLINE'].value_counts()

IS_ONLINE
1    6126
0     609
Name: count, dtype: int64

In [41]:
df.to_csv('insights.csv')

In [106]:
from geopy.distance import geodesic

# Define the epicenter
epicenter = (-3.745  , 127.752)
# Function to calculate distance
def is_within_radius(lat, lon, center, radius_km=100):
    return geodesic((lat, lon), center).km <= radius_km

# Filter rows within 100 km of the epicenter
area_filtered_df = df[
    df1.apply(lambda row: is_within_radius(row['latitude'], row['longitude'], epicenter), axis=1)
]


C:\Users\gokul\AppData\Local\Temp\ipykernel_21296\973289995.py:10: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  area_filtered_df = df[


In [107]:
len(area_filtered_df.NS_ID.unique())

0

In [93]:
area_filtered_df.IP.unique()

array(['113.212.113.31', '113.212.126.31', '103.68.0.44', '103.68.0.45'],
      dtype=object)

In [13]:
area_filtered_df.IP.value_counts()

IP
113.212.126.31     95
113.212.113.31     94
202.162.192.4      32
103.68.0.44        28
110.232.85.237     21
202.162.205.233    21
103.68.0.45        14
118.97.204.120      7
118.97.204.121      7
Name: count, dtype: int64

In [15]:
len(area_filtered_df.NS_ID.unique())

173

In [14]:
area_filtered_df.groupby('NS_ID').head(20)

,NS_ID,NAME,IP,latitude,longitude,LAST_SEEN,IS_ONLINE,IS_AUTH
209,28308873,ns2.wanxp.net.,113.212.126.31,0.5167,101.4417,2017-06-16 09:49:56,0,0
212,28307207,dns2.wanxp.net.,113.212.126.31,0.5167,101.4417,2017-06-16 09:49:08,1,1
223,28307322,dns1.wanxp.net.,113.212.113.31,0.5167,101.4417,2017-06-16 09:49:10,1,1
237,28308871,dns2.wanxp.net.,113.212.126.31,0.5167,101.4417,2017-06-16 09:49:56,0,0
255,28307244,ns1.wanxp.net.,113.212.113.31,0.5167,101.4417,2017-06-16 09:49:08,1,1
...,...,...,...,...,...,...,...,...
5980,28307564,ns2.wanxp.net.,113.212.126.31,0.5167,101.4417,2017-06-16 09:49:15,1,1
5982,28308813,ns1.wanxp.net.,113.212.113.31,0.5167,101.4417,2017-06-16 09:49:54,1,1
5983,28308874,dns1.wanxp.net.,113.212.113.31,0.5167,101.4417,2017-06-16 09:49:56,1,1
5984,28308872,ns1.wanxp.net.,113.212.113.31,0.5167,101.4417,2017-06-16 09:49:56,1,1


In [17]:
!pip install openai

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 27.3 MB/s eta 0:00:00


In [74]:
import requests
import pandas as pd

# Define the API endpoint and parameters
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
params = {
    "format": "geojson",
    "starttime": "2017-06-01",
    "endtime": "2017-08-31",
    "minlatitude": -11.0,  # Indonesia latitude range
    "maxlatitude": 6.0,
    "minlongitude": 95.0,  # Indonesia longitude range
    "maxlongitude": 141.0,
    "minmagnitude": 4.0  # Minimum magnitude to consider
}

# Fetch the data
response = requests.get(url, params=params)

# Check if the response is successful
if response.status_code == 200:
    data = response.json()

    # Extract features (earthquake events)
    features = data.get("features", [])
    
    # Create a DataFrame
    earthquakes = []
    for feature in features:
        properties = feature.get("properties", {})
        geometry = feature.get("geometry", {}).get("coordinates", [])
        earthquakes.append({
            "time": pd.to_datetime(properties.get("time"), unit="ms"),
            "latitude": geometry[1],
            "longitude": geometry[0],
            "depth_km": geometry[2],
            "magnitude": properties.get("mag"),
            "place": properties.get("place")
        })
    
    earthquake_df = pd.DataFrame(earthquakes)
    
    # Display the DataFrame
    print(earthquake_df.head())
else:
    print(f"Error fetching data: {response.status_code}")

# Save the DataFrame to a CSV file
earthquake_df.to_csv("indonesia_earthquakes_june_july_2017.csv", index=False)


                     time  latitude  longitude  depth_km  magnitude  \
0 2017-08-30 21:00:30.500   -2.7014   139.4862     10.00        4.2   
1 2017-08-30 20:56:18.440   -4.2799   130.3128     64.30        4.0   
2 2017-08-30 19:55:08.020   -1.1870   138.2956     32.84        4.3   
3 2017-08-30 19:47:43.430   -1.5367   138.3084     32.66        4.5   
4 2017-08-30 18:07:11.250    2.7868   122.6638    509.33        4.3   

                              place  
0    127 km W of Abepura, Indonesia  
1    186 km SE of Amahai, Indonesia  
2       246 km E of Biak, Indonesia  
3       250 km E of Biak, Indonesia  
4  252 km N of Gorontalo, Indonesia  


In [84]:
earthquake_df[earthquake_df['magnitude']>6]

,time,latitude,longitude,depth_km,magnitude,place
72,2017-08-13 03:08:10.560,-3.7682,101.6228,31.0,6.4,"71 km W of Bengkulu, Indonesia"


In [86]:
len(df.NS_ID.unique())

12207

In [46]:
df[df['NS_ID'].isnull()==True]

,NS_ID,NAME,IP,latitude,longitude,LAST_SEEN,IS_ONLINE,IS_AUTH


In [87]:
df[df['IS_ONLINE']==0]

,NS_ID,_c0,IP,latitude,longitude,ID,SEEN,AVG_RTT_MILLIS,IS_ONLINE,IS_AUTH,SERIAL,VERSION,EDNS0,CNAME,BOGUS,VALID_SOA,KEYS_VERIF,FOUND_WHERE
140,29142569,23166,103.56.148.24,-6.4000,106.8186,74367290,1498764098,NaN,0,0,0,NaN,0,0,0,2,2,2
142,29142588,23168,103.56.148.24,-6.4000,106.8186,74367295,1498764098,NaN,0,0,0,NaN,0,0,0,2,2,2
144,29143108,23184,103.56.148.24,-6.4000,106.8186,74367298,1498764099,NaN,0,0,0,NaN,0,0,0,2,2,2
146,29142553,23164,103.56.148.24,-6.4000,106.8186,74367309,1498764099,NaN,0,0,0,NaN,0,0,0,2,2,2
148,29142924,23178,103.56.148.24,-6.4000,106.8186,74367312,1498764099,NaN,0,0,0,NaN,0,0,0,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23439,32753804,19392,183.91.64.222,-6.2146,106.8451,129824862,1500419601,NaN,0,0,0,NaN,0,0,0,2,2,3
23441,32753820,19399,183.91.64.222,-6.2146,106.8451,129825809,1500419619,NaN,0,0,0,NaN,0,0,0,2,2,3
23444,32753818,19398,183.91.64.222,-6.2146,106.8451,129826054,1500419625,NaN,0,0,0,NaN,0,0,0,2,2,3
23446,32753816,19397,183.91.64.222,-6.2146,106.8451,129826373,1500419634,NaN,0,0,0,NaN,0,0,0,2,2,3


In [62]:
import pandas as pd
from geopy.distance import geodesic
from datetime import timedelta

# Load server and earthquake data
servers = df # Replace with your server data file
earthquakes = earthquake_df # Replace with your earthquake data file

# Ensure correct column names
print("Server Columns:", servers.columns)
print("Earthquake Columns:", earthquakes.columns)

# Convert `LAST_SEEN` to datetime
if 'LAST_SEEN' in servers.columns:
    servers['LAST_SEEN'] = pd.to_datetime(servers['LAST_SEEN'], errors='coerce')
else:
    raise KeyError("Column 'LAST_SEEN' not found in server data.")

# Convert `time` to datetime
if 'time' in earthquakes.columns:
    earthquakes['time'] = pd.to_datetime(earthquakes['time'], errors='coerce')
else:
    raise KeyError("Column 'time' not found in earthquake data.")

# Function to calculate distance between server and earthquake epicenter
def calculate_distance(row, quake):
    server_coords = (row['latitude'], row['longitude'])
    quake_coords = (quake['latitude'], quake['longitude'])
    return geodesic(server_coords, quake_coords).km

# Correlation: Add earthquake information to servers within 100km radius
correlated_data = []

for _, quake in earthquakes.iterrows():
    quake_time = quake['time']
    quake_coords = (quake['latitude'], quake['longitude'])

    for _, server in servers.iterrows():
        # Calculate distance
        if 'latitude' in server and 'longitude' in server:
            distance = geodesic((server['latitude'], server['longitude']), quake_coords).km

            # Check if within 100 km and after quake
            if distance <= 200 and server['LAST_SEEN'] >= quake_time and server['LAST_SEEN'] <= quake_time+ timedelta(10):
                print(distance)
                correlated_data.append({
                    'NS_ID': server.get('NS_ID', 'Unknown'),
                    'IS_ONLINE': server.get('IS_ONLINE', 'Unknown'),
                    'LAST_SEEN': server['LAST_SEEN'],
                    'Server_Lat': server['latitude'],
                    'Server_Lon': server['longitude'],
                    'Quake_Lat': quake['latitude'],
                    'Quake_Lon': quake['longitude'],
                    'Quake_Magnitude': quake['magnitude'],
                    'Distance_km': distance,
                    'Downtime_Hours': (server['LAST_SEEN'] - quake_time).total_seconds() / 3600
                })

# Create correlated dataframe
correlated_df = pd.DataFrame(correlated_data)

# Save correlated data
correlated_df.to_csv("correlated_earthquake_server_data.csv", index=False)

# Insights
total_servers_affected = len(correlated_df['NS_ID'].unique())
avg_downtime = correlated_df['Downtime_Hours'].mean()
most_impacted_quake = correlated_df.groupby(['Quake_Lat', 'Quake_Lon', 'Quake_Magnitude']).size().idxmax()

# Print insights
print(f"Total servers affected: {total_servers_affected}")
print(f"Average downtime (hours): {avg_downtime:.2f}")
print(f"Most impacted quake location (Lat, Lon, Magnitude): {most_impacted_quake}")


Server Columns: Index(['NS_ID', 'NAME', 'IP', 'latitude', 'longitude', 'LAST_SEEN',
       'IS_ONLINE', 'IS_AUTH'],
      dtype='object')
Earthquake Columns: Index(['time', 'latitude', 'longitude', 'depth_km', 'magnitude', 'place'], dtype='object')
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.9497922375977
162.949792237

In [60]:
correlated_df.head()

,NS_ID,IS_ONLINE,Server_Lat,Server_Lon,Quake_Lat,Quake_Lon,Quake_Magnitude,Distance_km,Downtime_Hours
0,53027998,1,-6.2146,106.8451,-7.4094,105.9824,5.2,162.949792,2.185653
1,53172442,1,-6.2146,106.8451,-7.4094,105.9824,5.2,162.949792,3.686764
2,53027823,1,-6.2146,106.8451,-7.4094,105.9824,5.2,162.949792,2.184819
3,53171982,1,-6.2146,106.8451,-7.4094,105.9824,5.2,162.949792,3.682319
4,53172401,1,-6.2146,106.8451,-7.4094,105.9824,5.2,162.949792,3.686208


In [ ]:
import pandas as pd


# Sort the dataframe by NS_ID and time (if there is a time column)
df = correlated_df.sort_values(by=['NS_ID', 'LAST_SEEN'])  # Replace 'LAST_SEEN' with the appropriate time column name

# Identify transitions from online (1) to offline (0) and vice versa
df['Transition'] = df.groupby('NS_ID')['IS_ONLINE'].diff()

# Count the servers that went offline and came back online
offline_to_online = df[(df['Transition'] == 1)]  # Transition from offline (0) to online (1)
online_to_offline = df[(df['Transition'] == -1)]  # Transition from online (1) to offline (0)

# Number of unique servers that went offline and came back online
servers_went_offline = online_to_offline['NS_ID'].nunique()
servers_back_online = offline_to_online['NS_ID'].nunique()

print(f"Number of servers that went offline: {servers_went_offline}")
print(f"Number of servers that came back online: {servers_back_online}")

# Display the transitions for detailed analysis
print("\nServers went offline:")
print(online_to_offline)

print("\nServers came back online:")
print(offline_to_online)


Number of servers that went offline: 0
Number of servers that came back online: 0

Servers went offline:
Empty DataFrame
Columns: [NS_ID, IS_ONLINE, LAST_SEEN, Server_Lat, Server_Lon, Quake_Lat, Quake_Lon, Quake_Magnitude, Distance_km, Downtime_Hours, Transition]
Index: []

Servers came back online:
Empty DataFrame
Columns: [NS_ID, IS_ONLINE, LAST_SEEN, Server_Lat, Server_Lon, Quake_Lat, Quake_Lon, Quake_Magnitude, Distance_km, Downtime_Hours, Transition]
Index: []


In [49]:
correlated_df.head()

""


In [20]:
!pip install tabulate

In [24]:
!pip install openai==0.28

  Attempting uninstall: openai
    Found existing installation: openai 1.55.0
    Uninstalling openai-1.55.0:
      Successfully uninstalled openai-1.55.0


In [28]:
area_filtered_df.to_csv('filtered_area1_eq.csv')

In [30]:
from datetime import datetime, timedelta
# Define earthquake timeframe
start_time = "2018-07-14 16:10:51"
end_time = "2018-08-21 23:59:59"
start_time = datetime.strptime("2018-07-14 16:10:51", "%Y-%m-%d %H:%M:%S")
end_time = datetime.strptime("2018-08-21 23:59:59", "%Y-%m-%d %H:%M:%S")
# Convert timestamp column to datetime

# Filter DataFrame for the earthquake period
time_filtered_df = area_filtered_df[(area_filtered_df['LAST_SEEN'] >= start_time-timedelta(20)) & 
                             (area_filtered_df['LAST_SEEN'] <= end_time+timedelta(10))]


In [40]:
df.LAST_SEEN.max()

Timestamp('2017-07-09 06:13:16')